In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_report, precision_recall_curve

np.random.seed(42); tf.random.set_seed(42)

In [2]:
FILE_PATH = r'/content/sleep_apnea_synthetic_data.csv'
df = pd.read_csv(FILE_PATH)

features = ['age','bmi','acc_mean','acc_std','hr_mean','hr_std','spo2_mean','spo2_min']
target = 'is_apnea_event'

n0, n1 = df[target].value_counts()
print(f"Data: {len(df):,} rows | Class 0: {n0:,} | Class 1: {n1:,} (Weight: {n0/n1:.2f})")
class_weight = {0: 1.0, 1: n0 / n1}

Data: 495,674 rows | Class 0: 397,915 | Class 1: 97,758 (Weight: 4.07)


In [3]:
train_df = df[df['subject_id'].isin(['S0000','S0001','S0002'])]
val_df   = df[df['subject_id'].isin(['S0003'])]
test_df  = df[df['subject_id'].isin(['S0004'])]

scaler = StandardScaler()
X_tr = scaler.fit_transform(train_df[features])
X_va = scaler.transform(val_df[features])
X_te = scaler.transform(test_df[features])
y_tr, y_va, y_te = train_df[target].values, val_df[target].values, test_df[target].values

SEQ = 20
def seq_gen(X, y):
    Xs, ys = [], []
    for i in range(SEQ, len(X)):
        Xs.append(X[i-SEQ:i]); ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_tr_s, y_tr_s = seq_gen(X_tr, y_tr)
X_va_s, y_va_s = seq_gen(X_va, y_va)
X_te_s, y_te_s = seq_gen(X_te, y_te)
print(f"Train: {X_tr_s.shape} | Val: {X_va_s.shape} | Test: {X_te_s.shape}")

Train: (14380, 20, 8) | Val: (4780, 20, 8) | Test: (4780, 20, 8)


In [4]:
def tcn_block(x, filters, kernel, dilation):
    res = x
    x = layers.Conv1D(filters, kernel, padding='causal', dilation_rate=dilation, activation='relu')(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.2)(x)
    if res.shape[-1] != filters: res = layers.Conv1D(filters, 1, padding='same')(res)
    return layers.Activation('relu')(layers.Add()([x, res]))

def build_tcn():
    inp = keras.Input(shape=(SEQ, len(features)))
    x = inp
    for d in [1,2,4,8,16]:
        x = tcn_block(x, 64, 3, d)
    x = layers.Lambda(lambda t: t[:, -1, :])(x)
    x = layers.Dense(64, activation='relu')(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(x))

model = build_tcn()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 20, 64)    │      1,600 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 20, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 20, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 20, 64)    │        576 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 64)    │          0 │ dropout[0][0],    │
│                     │                   │            │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 20, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 20, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 20, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 64)    │          0 │ dropout_1[0][0],  │
│                     │                   │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 20, 64)    │          0 │ add_1[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 20, 64)    │     12,352 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 64)    │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 20, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 20, 64)    │          0 │ dropout_2[0][0],  │
│                     │                   │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 20, 64)    │          0 │ add_2[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 20, 64)    │     12,352 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 64)    │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 59,137 (231.00 KB)

 Trainable params: 58,497 (228.50 KB)

 Non-trainable params: 640 (2.50 KB)

In [5]:
callbacks = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
model.fit(X_tr_s, y_tr_s, validation_data=(X_va_s, y_va_s),
          epochs=20, batch_size=128, class_weight=class_weight,
          callbacks=callbacks, verbose=1)
model.save('tcn_model.h5')

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 32s 144ms/step - AUC: 0.6435 - loss: 1.0183 - val_AUC: 0.6421 - val_loss: 0.6015
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 15s 96ms/step - AUC: 0.7309 - loss: 0.8392 - val_AUC: 0.6547 - val_loss: 0.5613
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - AUC: 0.7583 - loss: 0.8048 - val_AUC: 0.6722 - val_loss: 0.5481
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 91ms/step - AUC: 0.7691 - loss: 0.7931 - val_AUC: 0.6716 - val_loss: 0.5157
Epoch 5/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 10s 88ms/step - AUC: 0.7775 - loss: 0.7803 - val_AUC: 0.6695 - val_loss: 0.5131
Epoch 6/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - AUC: 0.7827 - loss: 0.7719 - val_AUC: 0.6719 - val_loss: 0.4884
Epoch 7/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 26s 148ms/step - AUC: 0.7914 - loss: 0.7612 - val_AUC: 0.6759 - val_loss: 0.5269
Epoch 8/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 17s 122ms/step - AUC: 0.7932 - loss: 0.7570 - val_AUC: 0.6683 - val_loss: 0.5253
Epoch 9/20
113/113 ━━━━━━━━━━━━━━━━━━

In [6]:
va_probs = model.predict(X_va_s).flatten()
te_probs = model.predict(X_te_s).flatten()

prec, rec, thresh = precision_recall_curve(y_va_s, va_probs)
f1s = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
best_th = thresh[np.argmax(f1s)] if len(thresh) > 0 else 0.5

te_preds = (te_probs >= best_th).astype(int)

print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"AUC-ROC  : {roc_auc_score(y_te_s, te_probs):.4f}")
print(f"F1-Score : {f1_score(y_te_s, te_preds):.4f}")
print(f"Threshold: {best_th:.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_te_s, te_preds))
print("\nClassification Report:")
print(classification_report(y_te_s, te_preds, target_names=['No Apnea', 'Apnea']))

150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

TEST SET PERFORMANCE
AUC-ROC  : 0.6396
F1-Score : 0.5722
Threshold: 0.528

Confusion Matrix:
[[1671 1046]
 [ 817 1246]]

Classification Report:
              precision    recall  f1-score   support

    No Apnea       0.67      0.62      0.64      2717
       Apnea       0.54      0.60      0.57      2063

    accuracy                           0.61      4780
   macro avg       0.61      0.61      0.61      4780
weighted avg       0.62      0.61      0.61      4780



In [8]:
from sklearn.metrics import accuracy_score

# Then, after you compute y_pred, add this:
accuracy = accuracy_score(y_te_s, te_preds)
print(f"✅ Accuracy   : {accuracy:.4f}")

✅ Accuracy   : 0.6103
